In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns


In [ ]:
df_data = pd.read_csv(r'C:/Users/Саша/OneDrive/Documents/stack-overflow-developer-survey-2025/survey_results_public.csv', low_memory=False)
df_description = pd.read_csv(r'C:/Users/Саша/OneDrive/Documents/stack-overflow-developer-survey-2025/survey_results_schema.csv')
print (df_data.shape)
df_data.head()

### Завдання 1. Підрахунок загальної кількості респондентів


In [ ]:
responents = df_data['ResponseId'].nunique() #кількість унікальних значень
print ('Загальна кількість респондентів в опитуванні: ', responents)

### Завдання 2. Аналіз повноти відповідей респондентів


In [ ]:
df_description['qname'] #список питань
questions = list (set(df_description['qname']).intersection(df_data.columns))
#set(df_description['qname']) - set(df_data.columns) #питання зі списку у файлі опису, які відсутні у файлі результатів
df_data_cleared = df_data[questions].dropna(axis=0, inplace=False) #підрахунок результатів без пропусків

print ('Кількість респондентів, які відповіли на всі запитання з опитування: ', df_data_cleared.shape[0])

### Завдання 3. Статистичний аналіз досвіду респондентів



In [ ]:
df_data.WorkExp.describe() #для перевірки

In [ ]:
pd.DataFrame([{
    'mean': round(df_data['WorkExp'].mean(), 2),
    'median': df_data['WorkExp'].median(),
    'mode': df_data['WorkExp'].mode()[0]
}])

### Завдання 4. Аналіз віддаленої роботи

In [ ]:
df_data.columns[df_data.apply(lambda col: col.astype(str).str.contains('remote', case=False, na=False)).any()] #пошук колонки з форматом роботи
df_data['RemoteWork'].unique() #перевірка правильності написання варіантів

In [ ]:
df_data_remote = df_data[df_data['RemoteWork']=='Remote']
df_data_remote
print ('Кількість респондентів, які працюють віддалено: ' ,df_data_remote['ResponseId'].nunique() )

### Завдання 5. Визначення популярності Python

In [ ]:
df_data.columns[df_data.isin(['Python']).any()] #колонки з мовами програмування

In [ ]:
df_python = df_data [df_data['LanguageHaveWorkedWith'].str.contains('Python', na=False)] #створення таблиці з Python-спеціалістами
df_python.to_csv('python_data.csv', index=False)

In [ ]:
works_with_python = df_data['LanguageHaveWorkedWith'].str.contains('Python', na=False).sum()
python_percent = (works_with_python / responents)*100
print ('Відсоток респондентів, які програмують на Python: ', round(python_percent, 2),'%')

### Завдання 6. Аналіз шляхів навчання програмуванню


In [ ]:
cols = [c for c in df_data.columns if df_data[c].astype(str).str.contains('Online Courses', na=False).any()]
cols # стовпчики з інформацією про способи навчання

In [ ]:
print ('Кількість респондентів, які навчалися програмувати через онлайн курси: ', \
df_data[df_data['LearnCode'].str.contains('Online Courses', na=False)].shape[0])

### Завдання 7. Географічний аналіз компенсації Python-розробників

In [ ]:
df_data.columns.isin(['Country']).any() #перевірка чи є колонка за назвою

In [ ]:
df_python_countries = df_python.dropna(subset=['ConvertedCompYearly'])
result = df_python.dropna(subset=['ConvertedCompYearly']).groupby('Country')[['ConvertedCompYearly']].agg(['mean', 'median'])

df_python_countries.to_csv('df_python_countries.csv', index=False)
pd.DataFrame(result).sort_values(by=('ConvertedCompYearly', 'mean'), ascending=False).round()

### Завдання 8. Аналіз освіти найбільш оплачуваних спеціалістів


In [ ]:
most_paid = df_data.dropna(subset=['ConvertedCompYearly']).sort_values(by='ConvertedCompYearly', ascending=False)[:5]
pd.DataFrame(most_paid['EdLevel'])

### Завдання 9. Аналіз популярності Python по віковим категоріям


In [ ]:
result = (df_python.groupby('Age')['ResponseId'].count() / df_data.groupby('Age')['ResponseId'].count() * 100).round(2)
pd.DataFrame(result).rename(columns={"ResponseId": "Response_python_percantage, %"})

### Завдання 10. Аналіз індустрій серед високооплачуваних віддалених працівників


In [ ]:
df_data.columns.isin(['Industry']).any() #перевірка чи є колонка за назвою
#df_data['Industry'].unique() #перевірка варіантів індустрії

In [ ]:
percentile_75 = df_data['ConvertedCompYearly'].quantile(0.75)
df_most_paid = df_data_remote[df_data_remote['ConvertedCompYearly']>=percentile_75]
industry_most_paid = df_most_paid['Industry'].value_counts()
pd.DataFrame(industry_most_paid)